In [1]:
sql = "SELECT COUNT(COUNTEID) FROM ( SELECT T1.EmployeeID AS COUNTEID FROM Employees AS T1 INNER JOIN EmployeeTerritories AS T2 ON T1.EmployeeID = T2.EmployeeID WHERE T1.Country = 'UK' GROUP BY T1.EmployeeID HAVING COUNT(T2.TerritoryID) > 4 ) T1"
from sql_metadata import Parser
used_tables = Parser(sql).tables
print(used_tables)

['Employees', 'EmployeeTerritories']


In [ ]:
import json
import os

def is_placeholder(entry: dict) -> bool:
    if not isinstance(entry, dict):
        return True
    inp = entry.get("input", "")
    outs = entry.get("outputs", None)
    if isinstance(inp, str) and inp.startswith("[PLACEHOLDER]"):
        return True
    if isinstance(outs, list) and len(outs) == 0:
        return True
    return False

def clean_placeholders(input_path: str, output_path: str, overwrite: bool = False):
    kept = 0
    removed = 0
    kept_lines = []

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                # Lines that cannot be parsed are considered invalid and discarded directly
                removed += 1
                continue

            if is_placeholder(obj):
                removed += 1
            else:
                kept_lines.append(json.dumps(obj, ensure_ascii=False))
                kept += 1

    tmp_path = output_path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        for l in kept_lines:
            f.write(l + "\n")

    if overwrite and os.path.abspath(output_path) == os.path.abspath(input_path):
        # Overwrite in place
        os.replace(tmp_path, output_path)
    else:
        # Write to separate output file
        os.replace(tmp_path, output_path)

    print(f"Kept: {kept}, Removed: {removed}, Output: {output_path}")

if __name__ == "__main__":
    input_file = "/Users/wuyuyang/Code/schlink/BIRD_Data/data/embedding_training_data.json"
    output_file = "/Users/wuyuyang/Code/schlink/BIRD_Data/data/embedding_training_data.json"
    clean_placeholders(input_file, output_file, overwrite=True)

Kept: 9394, Removed: 34, Output: /Users/wuyuyang/Code/schlink/BIRD_Data/data/embedding_training_data.json
